## Code Walkthrough
_Study Unit: ARI5118 - Deep Learning for Computer Vision_<br>
_Topic: CNN Feature Visualisation and DeepDream_<br>
_Author: David Farrugia_

**Notebook Description**

## 0. Setup

In [1]:
import time
import random
from pathlib import Path

import torch
from torchvision import models, transforms
from PIL import Image

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">> Using device: {DEVICE}")

>> Using device: cuda


In [3]:
start_time = time.time()

## 1. Config

In [ ]:
CONFIG = {
    # DIRECTORIES
    "input_imgs_dir": Path("./simulator/assets/imgs/input_imgs"),
    "output_dir": Path("./simulator/assets/imgs/outputs"),

    # IMAGE SETTINGS
    "IMAGE_SIZE": 224,

    # FEATURE MAPS
    "NUM_FILTERS": 10,

    # ACTIVATION MAXIMISATION
    "ACT_MAX_STEPS": 100,
    "ACT_MAX_STEP_SIZE": [0.001, 0.01, 0.1],

    # REGULARISATION
    "L2_LAMBDA": 1e-5,

    # DEEPDREAM
    "DEEPDREAM_STEPS": 40,
    "DEEPDREAM_NOISE": 0.01,
    "DEEPDREAM_STEP_SIZE": 0.001,
    "DEEPDREAM_OCTAVES": [2, 3, 4],
    "DEEPDREAM_OCTAVE_SCALE": [0.6, 0.8, 1.0, 1.2, 1.4],
    
    # BOOL FLAGS - Executions had to be done in stages to avoid long runtimes and GPU memory issues
    "GET_FEATURE_MAPS": False,
    "RUN_ACT_MAX": False,
    "RUN_DEEPDREAM": True
}

### 1.1. Validating Directories

In [5]:
INPUT_DIR = CONFIG["input_imgs_dir"]
OUTPUT_DIR = CONFIG["output_dir"]

if not INPUT_DIR.exists():
    raise FileNotFoundError(f">> [ERROR] Input image folder not found: {INPUT_DIR.resolve()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Loading Input Images Paths

In [6]:
IMAGE_PATHS = []

# Load all images from the directory
image_extensions = ['*.jpg', '*.jpeg', '*.png']
images = []

for ext in image_extensions:
    IMAGE_PATHS.extend(INPUT_DIR.glob(ext))

if len(IMAGE_PATHS) == 0:
    raise ValueError(f">> [ERROR] No images found in: {INPUT_DIR.resolve()}")

print(f">> --- [Found {len(IMAGE_PATHS)} input images] ---:")
for path in IMAGE_PATHS:
    print(f">> {path.name}")

>> --- [Found 5 input images] ---:
>> Black-Cat.png
>> Golden-Retriever.png
>> mountains.png
>> Parrot.png
>> Tokyo_Shibuya.png


## 3. Load pretrained model

In [7]:
weights = models.VGG16_Weights.DEFAULT
model = models.vgg16(weights=weights).to(DEVICE)
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### 3.1 Select Convolution Layers

In [8]:
# First 5 convolutional layers
TARGET_LAYERS = {
    "conv1": model.features[0],
    "conv2": model.features[2],
    "conv3": model.features[5],
    "conv4": model.features[7],
    "conv5": model.features[10],
}

## 4. Image Transforms and Helper Functions

### 4.1. Image Transforms

In [9]:
preprocess = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

plain_transform = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.ToTensor()
])

### 4.2. Helper Functions

In [10]:
def load_image(path):
    return Image.open(path).convert("RGB")

In [11]:
def tensor_to_display_img(tensor):
    """
    Converts a tensor to a displayable image.
    Handles tensors shaped [3, H, W].
    """
    tensor = tensor.detach().cpu().clone()

    # Undo ImageNet normalization approximately if needed
    if tensor.min() < 0:
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        tensor = tensor * std + mean

    tensor = tensor.clamp(0, 1)
    return tensor.permute(1, 2, 0).numpy()

In [12]:
def save_tensor_image(tensor, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)

    img = tensor_to_display_img(tensor)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close()


In [13]:
def save_feature_map(feature_map, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)

    feature_map = feature_map.detach().cpu()
    feature_map = feature_map - feature_map.min()
    feature_map = feature_map / (feature_map.max() + 1e-8)

    plt.figure(figsize=(4, 4))
    plt.imshow(feature_map.numpy(), cmap="viridis")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close()

## 5. Register activation hooks


In [14]:
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output
    return hook

hooks = []

for name, layer in TARGET_LAYERS.items():
    hook = layer.register_forward_hook(get_activation(name))
    hooks.append(hook)

print(">> Hooks registered for:", list(TARGET_LAYERS.keys()))

>> Hooks registered for: ['conv1', 'conv2', 'conv3', 'conv4', 'conv5']


## 6. Generate feature maps

In [15]:
if CONFIG["GET_FEATURE_MAPS"]:
    
    def generate_feature_maps():
        print("\n>> --- [Generating feature maps...] --- :")

        for image_path in IMAGE_PATHS:
            image_name = image_path.stem

            image = load_image(image_path)
            input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                _ = model(input_tensor)

            for layer_name, activation in activations.items():
                activation = activation[0]  # [channels, H, W]

                num_channels = min(CONFIG["NUM_FILTERS"], activation.shape[0])

                for channel_idx in range(num_channels):
                    feature_map = activation[channel_idx]

                    save_path = (OUTPUT_DIR / "feature_maps" / image_name / layer_name / f"filter_{channel_idx + 1}.png")
                    save_feature_map(feature_map, save_path)

            print(f">> [OK] Saved feature maps for: {image_name}")
        print(">> [DONE] Feature map generation complete.")

    generate_feature_maps()
else:
    print(">> [SKIP] Feature map generation skipped due to config flag.")

>> [SKIP] Feature map generation skipped due to config flag.


## 7. Activation Maximisation

In [16]:
if CONFIG["RUN_ACT_MAX"]:
    def activation_maximisation(layer_name, filter_idx, step_size, use_l2=False):
        """
        Generates an image that maximises a chosen filter/channel activation.
        Starts from random noise.
        """

        if layer_name not in TARGET_LAYERS:
            raise ValueError(f"Unknown layer: {layer_name}")

        # Random image input
        input_img = torch.rand(
            1, 3,
            CONFIG["IMAGE_SIZE"],
            CONFIG["IMAGE_SIZE"],
            device=DEVICE,
            requires_grad=True
        )

        optimizer = torch.optim.Adam(
            [input_img],
            lr=step_size
        )

        for step in range(CONFIG["ACT_MAX_STEPS"]):
            optimizer.zero_grad()

            activations.clear()
            _ = model(input_img)

            target_activation = activations[layer_name][0, filter_idx].mean()

            loss = -target_activation

            if use_l2:
                loss = loss + CONFIG["L2_LAMBDA"] * torch.mean(input_img ** 2)

            loss.backward()
            optimizer.step()

            with torch.no_grad():
                input_img.clamp_(0, 1)

        return input_img[0].detach()

In [17]:
if CONFIG["RUN_ACT_MAX"]:
    
    def generate_activation_maximisation_outputs():
        print("\n>> --- [Generating activation maximisation outputs...] --- :")

        for step_size in CONFIG["ACT_MAX_STEP_SIZE"]:
            for use_l2 in [False, True]:

                reg_folder = ("l2_regularisation" if use_l2 else "no_regularisation")

                for layer_name in TARGET_LAYERS.keys():
                    for filter_idx in range(CONFIG["NUM_FILTERS"]):

                        img = activation_maximisation(layer_name, filter_idx, step_size, use_l2)

                        save_path = (OUTPUT_DIR / "activation_maximisation" / reg_folder / f"step_{step_size}" 
                                    / layer_name / f"filter_{filter_idx + 1}.png")

                        save_tensor_image(img, save_path)

                    print(f">> [OK] Saved {layer_name} | step={step_size} | L2={use_l2}")
            print()
        print(">> [DONE] Activation maximisation complete.")
    
    generate_activation_maximisation_outputs()
else:
    print(">> [SKIP] Activation maximisation skipped due to config flag.")

>> [SKIP] Activation maximisation skipped due to config flag.


## 8. DeepDream

In [18]:
if CONFIG["RUN_DEEPDREAM"]:

    def deepdream_step(input_img, layer_name):
        """
        Performs one DeepDream optimisation pass on an image tensor.
        """

        input_img.requires_grad_(True)

        for step in range(CONFIG["DEEPDREAM_STEPS"]):
            model.zero_grad()

            activations.clear()
            _ = model(input_img)

            loss = activations[layer_name].mean()
            loss.backward()

            with torch.no_grad():
                grad = input_img.grad

                grad = grad / (grad.std() + 1e-8)

                input_img += CONFIG["DEEPDREAM_STEP_SIZE"] * grad
                input_img.clamp_(0, 1)

                input_img.grad.zero_()

        return input_img.detach()

In [19]:
if CONFIG["RUN_DEEPDREAM"]:

    def resize_tensor(tensor, size):
        return torch.nn.functional.interpolate(tensor, size=size, mode="bilinear",align_corners=False)

In [20]:
if CONFIG["RUN_DEEPDREAM"]:
    
    def deepdream(image_path, layer_name, octaves, octave_scale):
        image = load_image(image_path)
        img_tensor = plain_transform(image).unsqueeze(0).to(DEVICE)

        original_size = CONFIG["IMAGE_SIZE"]

        octave_sizes = []

        size = original_size
        for _ in range(octaves):
            octave_sizes.append(int(size))
            size = size / octave_scale

        octave_sizes = list(reversed(octave_sizes))

        dream = img_tensor
        # This was done to ensure the original image structure stays influential
        dream += CONFIG["DEEPDREAM_NOISE"] * torch.randn_like(dream)
        dream = dream.clamp(0, 1)

        for octave_size in octave_sizes:
            dream = resize_tensor(dream, (octave_size, octave_size))
            dream = deepdream_step(dream, layer_name)

        dream = resize_tensor(dream, (CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"]))

        return dream[0].detach()

In [21]:
if CONFIG["RUN_DEEPDREAM"]:
    
    def generate_deepdream_outputs():
        print("\n>> --- [Generating DeepDream outputs...] --- :")

        for image_path in IMAGE_PATHS:
            image_name = image_path.stem
            
            for octaves in CONFIG["DEEPDREAM_OCTAVES"]:
                for scale in CONFIG["DEEPDREAM_OCTAVE_SCALE"]:

                    for layer_name in TARGET_LAYERS.keys():
                        dream_img = deepdream(image_path, layer_name, octaves, scale)

                        save_path = (OUTPUT_DIR / "deepdream" / image_name / f"octaves_{octaves}" / f"scale_{scale}" / layer_name / "dream.png")

                        save_tensor_image(dream_img, save_path)

                    print(f">> [OK] Saved {image_name} | oct={octaves} | scale={scale}")
            print()
        print(">> [DONE] DeepDream generation complete.")

    generate_deepdream_outputs()
else:
    print(">> [SKIP] DeepDream generation skipped due to config flag.")


>> --- [Generating DeepDream outputs...] --- :
>> [OK] Saved Black-Cat | oct=2 | scale=0.6
>> [OK] Saved Black-Cat | oct=2 | scale=0.8
>> [OK] Saved Black-Cat | oct=2 | scale=1.0
>> [OK] Saved Black-Cat | oct=2 | scale=1.2
>> [OK] Saved Black-Cat | oct=2 | scale=1.4
>> [OK] Saved Black-Cat | oct=3 | scale=0.6
>> [OK] Saved Black-Cat | oct=3 | scale=0.8
>> [OK] Saved Black-Cat | oct=3 | scale=1.0
>> [OK] Saved Black-Cat | oct=3 | scale=1.2
>> [OK] Saved Black-Cat | oct=3 | scale=1.4
>> [OK] Saved Black-Cat | oct=4 | scale=0.6
>> [OK] Saved Black-Cat | oct=4 | scale=0.8
>> [OK] Saved Black-Cat | oct=4 | scale=1.0
>> [OK] Saved Black-Cat | oct=4 | scale=1.2
>> [OK] Saved Black-Cat | oct=4 | scale=1.4

>> [OK] Saved Golden-Retriever | oct=2 | scale=0.6
>> [OK] Saved Golden-Retriever | oct=2 | scale=0.8
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.0
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.2
>> [OK] Saved Golden-Retriever | oct=2 | scale=1.4
>> [OK] Saved Golden-Retriever | oct

## 9. Clean up hooks and timing

In [22]:
for hook in hooks:
    hook.remove()

In [23]:
elapsed_time = time.time() - start_time
print(f">> Elapsed time: {elapsed_time:.3f} seconds")

>> Elapsed time: 603.016 seconds
